In [1]:
!pip install datasets transformers evaluate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

In [3]:
# glue ichidan sst2 datasetini olamiz
dataset = load_dataset("glue", "sst2")

# train dan 2000 ta, validation dan 400 ta sample olamiz
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["validation"].shuffle(seed=42).select(range(400))

print("Train size:", len(small_train))
print("Test size:", len(small_test))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Train size: 2000
Test size: 400


In [4]:
# DistilBERT tokenizerini yuklaymiz
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# Matnlarni tokenlarga aylantiramiz
# max_length=128 qilib cheklaymiz
def preprocess_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
#sentence → input text
#truncation=True → uzun matnni kesadi
#padding="max_length" → hamma inputni 128 ga teng qiladi
#max_length=128 → maksimal uzunlik

In [6]:
# preprocess funksiyasini datasetga qo‘llaymiz
tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [7]:
# label ustunini labels deb o‘zgartiramiz
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# PyTorch formatiga o‘tkazamiz
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [8]:
# DistilBERT asosida classification model yuklaymiz
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# Model natijasini accuracy bilan baholaymiz
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [10]:
# Trening sozlamalarini beramiz
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    logging_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    load_best_model_at_end=True,
    seed=42,
    report_to="none"
)

In [11]:
# Trainer modelni o‘qitish va baholashni boshqaradi
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

In [12]:
# Modelni train qilamiz
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.334960,0.384354,0.837500
2,0.165504,0.345133,0.880000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=250, training_loss=0.31189248275756837, metrics={'train_runtime': 67.1946, 'train_samples_per_second': 59.529, 'train_steps_per_second': 3.721, 'total_flos': 132467398656000.0, 'train_loss': 0.31189248275756837, 'epoch': 2.0})

In [13]:
# Test datasetda natijani ko‘ramiz
results = trainer.evaluate()
print(results)

{'eval_loss': 0.3451332151889801, 'eval_accuracy': 0.88, 'eval_runtime': 1.5613, 'eval_samples_per_second': 256.194, 'eval_steps_per_second': 16.012, 'epoch': 2.0}


In [15]:
# Yangi gap berib prediction qilamiz
sample_text = "This movie was really amazing and enjoyable."

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True)

# inputs ni model device ga o'tkazamiz
inputs = {key: value.to(model.device) for key, value in inputs.items()}

outputs = model(**inputs)

prediction = np.argmax(outputs.logits.detach().cpu().numpy(), axis=1)[0]

if prediction == 1:
    print("Positive")
else:
    print("Negative")

Positive


Bu kod matn sentimentini (positive/negative) aniqlaydigan modelni yaratadi va o‘rgatadi.

Jarayon qisqacha:
Dataset yuklanadi → SST-2 (matn + label)
Kichraytiriladi → train (2000), test (400)
Tokenizatsiya → matn → raqam (input_ids)
Model yuklanadi → DistilBERT (classification)
Train qilinadi → model o‘rganadi
Baholanadi → accuracy hisoblanadi
Prediction qilinadi → yangi matnni baholaydi
Eng muhim g‘oya:
#

Model matnni raqamga aylantirib, shu asosda positive yoki negative ekanini aniqlaydi.


1 qatorlik summary:

Text → Tokenizer → Model → Prediction (0 yoki 1)
